In [0]:
import sys
import os
import pandas as pd
import joblib

# Configuration of modules and SRC folders path 
sys.path.append(os.path.abspath("../src"))

# Import Core Functions 
import importlib
from nlp import nlp_preferences
from recommender import recommender_engine
from utils import explanation_generator
from chatbot import chatbot_flow

# Reload modules to catch changes
importlib.reload(nlp_preferences)
importlib.reload(recommender_engine)
importlib.reload(explanation_generator)
importlib.reload(chatbot_flow)

# Import Specific Functions
from nlp.nlp_preferences import extract_preferences
from recommender.recommender_engine import recommend_on_the_fly
from utils.explanation_generator import generate_explanation
from chatbot.chatbot_flow import initialize_conversation_state

In [0]:

# Load of dataset cleaned and trained modules (Vectorizar and Matrix)
movies_df = pd.read_csv("../data/movies_final.csv")
# movies_df = spark.read.table("workspace.datasets.movies_final").toPandas()
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")
tfidf_matrix = joblib.load("../models/tfidf_matrix.pkl")

print(f"✅ Dataset Loaded: {movies_df.shape} Movies.")

In [0]:
def test_cineassist_pipeline(user_input, verbose=True):
    """Desktop test of the Whole PipeLine CineAssist.
    
    Args:
        user_input: Ask the user a question about movies (string)
        verbose: If True, detailed information of each stage and print the results.
    """
    print(f"\n{'='*80}")
    print(f"DESKTOP - PIPELINE CINEASSIST")
    print(f"{'='*80}")
    print(f"📝 User input: '{user_input}'\n")
    
    # ==== FASE 1: INICIALIZACIÓN ====
    if verbose:
        print("[FASE 1] Initializing conversation state...")
    state_dict = initialize_conversation_state()
    if verbose:
        print(f"   ✓ Inicial State: {state_dict}\n")
    
    # ==== FASE 2: PREFERENCES EXTRACTION (NLP) ====
    if verbose:
        print("[FASE 2] Extracting preferences from the text...")
    prefs = extract_preferences(user_input)
    state_dict.update(prefs)
    if verbose:
        print(f"   ✓ Genres detected: {prefs.get('genres', [])}")
        print(f"   ✓ Language: {prefs.get('language', 'No specified')}")
        print(f"   ✓ Year: {prefs.get('year', 'No specified')}")
        print(f"   ✓ Mood: {prefs.get('mood', 'No specified')}")
        print(f"   ✓ Min Rating: {prefs.get('rating', 'No specified')}")
        print(f"   ✓ State complete: {state_dict}\n")
    
    # ==== FASE 3: VALIDATION (CONTROL LOGIC) ====
    if verbose:
        print("[Stage 3] Validating enought information to process...")
    if not state_dict.get('genres'):
        print("   ❌ FAIL: No genre detected")
        print("   ⚠️ Bot will answer: 'I didn't find any genre. ¿what kind of movies would U like?'")
        return None
    if verbose:
        print(f"   ✓ Validation OK: {len(state_dict.get('genres', []))} genre detected\n")
    
    # ==== STAGE 4: QUERY GENERATION ====
    if verbose:
        print("[STAGE 4] Building search query ...")
    query_text = f"{user_input} {' '.join(state_dict['genres'])} {state_dict.get('mood', '')}"
    if verbose:
        print(f"   ✓ Query generated: '{query_text}'\n")
    
    # ==== STAGE 5: RECOMENDATION ENGINE ====
    if verbose:
        print("[STAGE 5] Executing recommendation engine...")
        print(f"   - Vectorizing query...")
        print(f"   - Calculating similarity with {len(movies_df)} movies...")
        print(f"   - Aplying filters: language={state_dict.get('language')}, year={state_dict.get('year')}, rating>={state_dict.get('rating')}")
    
    # Calling the recommended function, query, df, tfidf matrixs, and state of the query dict 
    recommendations = recommend_on_the_fly(query_text, movies_df, vectorizer, tfidf_matrix, state_dict)
    
    if verbose:
        print(f"   ✓ {len(recommendations)} recommendations generated\n")
    
    # ==== STAGE 6: GENERATE EXPLANATION ====
    if verbose:
        print("[STAGE 6] Generating perzonalized explanations...\n")
    
    print(f"{'='*80}")
    print(f"🎬 RESULTS - TOP {len(recommendations)} RECOMMENDATIONS")
    print(f"{'='*80}\n")
    
    for idx, (_, movie) in enumerate(recommendations.iterrows(), 1):
        explanation = generate_explanation(movie, state_dict)
        
        print(f"[{idx}] {movie['title']}")
        print(f"    ⭐ Rating: {movie['vote_average']:.1f} | Similitud: {movie['similarity_score']:.3f}")
        if verbose and 'release_date' in movie:
            print(f"    📅 Fecha: {movie['release_date']}")
        if verbose and 'genres_list' in movie:
            print(f"    🎭 Géneros: {movie['genres_list']}")
        print(f"    💡 {explanation}")
        print()
    
    print(f"{'='*80}")
    print("TEST END")
    print(f"{'='*80}\n")
    
    return recommendations

In [0]:
import sys
import os
import importlib
import pandas as pd

sys.path.append(os.path.abspath("../src"))
from metrics import metrics_test as test
importlib.reload(test)
from metrics.metrics_test import accuracy

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

state_dict = initialize_conversation_state()
tfidf_matrix_updated = vectorizer.transform(movies_df['combined_features'])

user_query="animation story about talking toys"
results = accuracy(recommend_on_the_fly(user_query, movies_df, vectorizer, tfidf_matrix_updated, state_dict),user_query)
print("".center(80, "="	))
print(f"For this User Query: \"{user_query}\"")
print(f"Accuracy: {results}")
print("".center(80, "="	))


In [0]:
# Scenery 0: Interactive search with validation loop
# If no genres are detected, it will ask again

# Ensure movies_df matches tfidf_matrix dimensions
if len(movies_df) != tfidf_matrix.shape[0]:
    movies_df = movies_df.head(tfidf_matrix.shape[0])

result = None
while result is None:
    input_text = input("Enter your question: ")
    result = test_cineassist_pipeline(
        input_text,
        verbose=False
    )
    
    if result is None:
        print("\n⚠️ Please try again with more specific information (mention a genre like comedy, drama, action, etc.).\n")

print("\n✅ Recommendation completed successfully!")

In [0]:
# Scenery 0: Interactive search with validation loop
# If no genres are detected, it will ask again

# Ensure movies_df matches tfidf_matrix dimensions
if len(movies_df) != tfidf_matrix.shape[0]:
    movies_df = movies_df.head(tfidf_matrix.shape[0])

result = None
while result is None:
    input_text = input("Enter your question: ")
    result = test_cineassist_pipeline(
        input_text,
        verbose=False
    )
    
    if result is None:
        print("\n⚠️ Please try again with more specific information (mention a genre like comedy, drama, action, etc.).\n")

print("\n✅ Recommendation completed successfully!")

In [0]:
# Scenery 0: Interactive search with validation loop
# If no genres are detected, it will ask again

# Ensure movies_df matches tfidf_matrix dimensions
if len(movies_df) != tfidf_matrix.shape[0]:
    movies_df = movies_df.head(tfidf_matrix.shape[0])

result = None
while result is None:
    input_text = input("Enter your question: ")
    result = test_cineassist_pipeline(
        input_text,
        verbose=False
    )
    
    if result is None:
        print("\n⚠️ Please try again with more specific information (mention a genre like comedy, drama, action, etc.).\n")

print("\n✅ Recommendation completed successfully!")